# Oferta: validación de la Normal mensual

Se valida el supuesto de que la oferta diaria de cada mes del año sigue una distribución Normal (`METODOLOGIA.md` §3), que es la que usa el modelo para generar escenarios (`codigo/distribuciones.py`). Todos los tests usan un nivel de significancia del 5% y, como se repiten en 12 meses, se corrigen por comparaciones múltiples con el método de Holm.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.diagnostic import lilliefors, normal_ad
from statsmodels.stats.multitest import multipletests

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "codigo").is_dir())
sys.path.insert(0, str(RAIZ))

from codigo import cargador_datos, distribuciones
from codigo import configuracion as cfg

ALFA = 0.05
MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]

oferta = cargador_datos.cargar_oferta_historica()
oferta["mes"] = oferta["fecha"].dt.month
oferta["anio"] = oferta["fecha"].dt.year
normales = distribuciones.ajustar_distribuciones_oferta(oferta)
por_mes = {m: g["oferta"].to_numpy(dtype=float) for m, g in oferta.groupby("mes")}


def resumen_tests(tabla, columnas):
    return pd.DataFrame({
        "valor-p mínimo": [tabla[c].min() for c in columnas],
        "rechazos sin corregir": [int((tabla[c] < ALFA).sum()) for c in columnas],
        "rechazos con Holm": [int(multipletests(tabla[c], alpha=ALFA, method="holm")[0].sum()) for c in columnas],
    }, index=columnas)


print(f"{len(oferta)} días; entre {min(map(len, por_mes.values()))} y {max(map(len, por_mes.values()))} observaciones por mes")

## 1. Tests de normalidad por mes

Se aplican tres tests a la oferta de cada mes (H0: los datos son normales): Shapiro-Wilk, Anderson-Darling y Kolmogorov-Smirnov con la corrección de Lilliefors, que corresponde cuando la media y la desviación se estiman de los mismos datos. Se reportan también la asimetría y el exceso de curtosis (0 en una Normal).

In [ ]:
tests = pd.DataFrame({
    m: {
        "n": len(x),
        "media": normales[m].media,
        "desviación": normales[m].desviacion,
        "asimetría": stats.skew(x),
        "exceso de curtosis": stats.kurtosis(x),
        "Shapiro-Wilk": stats.shapiro(x).pvalue,
        "Anderson-Darling": normal_ad(x)[1],
        "Lilliefors": lilliefors(x, dist="norm")[1],
    }
    for m, x in por_mes.items()
}).T
tests.index = MESES
tests["n"] = tests["n"].astype(int)
tests.round(3)

In [ ]:
resumen_tests(tests, ["Shapiro-Wilk", "Anderson-Darling", "Lilliefors"]).round(3)

**Conclusión:** la Normal no se rechaza en ningún mes una vez corregido por comparaciones múltiples. Sin corregir, sólo abril muestra rechazos (Anderson-Darling y Lilliefors, no Shapiro-Wilk), asociados a una asimetría negativa leve.

## 2. Revisión gráfica

Se compara cada mes con su Normal ajustada mediante histogramas y gráficos Q-Q (si los datos son normales, los puntos siguen la recta).

In [ ]:
fig, ejes = plt.subplots(3, 4, figsize=(12, 7.5), sharey=True)
for eje, (m, x) in zip(ejes.flat, por_mes.items()):
    d = normales[m]
    eje.hist(x, bins=12, density=True, color="0.8", edgecolor="white")
    rejilla = np.linspace(d.media - 4 * d.desviacion, d.media + 4 * d.desviacion, 200)
    eje.plot(rejilla, stats.norm.pdf(rejilla, d.media, d.desviacion), color="C3")
    eje.set_title(MESES[m - 1], fontsize=10)
fig.suptitle("Oferta diaria por mes y Normal ajustada")
plt.tight_layout()
plt.show()

In [ ]:
fig, ejes = plt.subplots(3, 4, figsize=(12, 8), sharex=True, sharey=True)
for eje, (m, x) in zip(ejes.flat, por_mes.items()):
    d = normales[m]
    teoricos, observados = stats.probplot((x - d.media) / d.desviacion, dist="norm", fit=False)
    eje.scatter(teoricos, observados, s=8, color="C0")
    eje.plot([-3, 3], [-3, 3], color="C3", lw=1)
    eje.set_title(MESES[m - 1], fontsize=10)
fig.suptitle("Gráficos Q-Q por mes (oferta estandarizada)")
fig.supxlabel("cuantil teórico de la Normal")
fig.supylabel("cuantil observado")
plt.tight_layout()
plt.show()

**Conclusión:** los histogramas calzan con la Normal ajustada y los puntos de los Q-Q siguen la recta en los 12 meses, sin desviaciones sistemáticas en las colas.

## 3. Comparación con otras distribuciones

Se ajustan por máxima verosimilitud una Gamma y una lognormal (ambas con 2 parámetros, igual que la Normal) y se comparan por AIC: el menor valor indica el mejor ajuste, y diferencias menores a 2 se consideran equivalentes.

In [ ]:
def aic(log_verosimilitud, n_parametros=2):
    return 2 * n_parametros - 2 * log_verosimilitud


filas = {}
for m, x in por_mes.items():
    d = normales[m]
    forma_g, _, escala_g = stats.gamma.fit(x, floc=0)
    forma_l, _, escala_l = stats.lognorm.fit(x, floc=0)
    filas[MESES[m - 1]] = {
        "Normal": aic(stats.norm.logpdf(x, d.media, d.desviacion).sum()),
        "Gamma": aic(stats.gamma.logpdf(x, forma_g, 0, escala_g).sum()),
        "Lognormal": aic(stats.lognorm.logpdf(x, forma_l, 0, escala_l).sum()),
    }
tabla_aic = pd.DataFrame(filas).T
tabla_aic["mejor"] = tabla_aic[["Normal", "Gamma", "Lognormal"]].idxmin(axis=1)
tabla_aic["Normal − mejor"] = tabla_aic["Normal"] - tabla_aic[["Normal", "Gamma", "Lognormal"]].min(axis=1)

print("Meses en que cada distribución tiene el menor AIC:", tabla_aic["mejor"].value_counts().to_dict())
print("AIC sumado sobre los 12 meses:", tabla_aic[["Normal", "Gamma", "Lognormal"]].sum().round(1).to_dict())
tabla_aic.round(2)

**Conclusión:** la Normal tiene el menor AIC en 8 de los 12 meses y también en el total. En los 4 meses restantes la diferencia con la mejor alternativa es menor a 2, es decir, los ajustes son equivalentes. Ninguna alternativa justifica reemplazar la Normal.

## 4. Homogeneidad entre años

Cada Normal mensual junta los datos de 2022, 2023 y 2024. Se verifica que eso sea válido comparando, dentro de cada mes, las medias entre años (ANOVA y Kruskal-Wallis) y las varianzas (Levene). H0: los tres años son iguales.

In [ ]:
filas = {}
for m, g in oferta.groupby("mes"):
    por_anio = [h["oferta"].to_numpy(dtype=float) for _, h in g.groupby("anio")]
    filas[MESES[m - 1]] = {
        **{f"media {a}": h["oferta"].mean() for a, h in g.groupby("anio")},
        **{f"desv. {a}": h["oferta"].std(ddof=0) for a, h in g.groupby("anio")},
        "ANOVA": stats.f_oneway(*por_anio).pvalue,
        "Kruskal-Wallis": stats.kruskal(*por_anio).pvalue,
        "Levene": stats.levene(*por_anio).pvalue,
    }
homogeneidad = pd.DataFrame(filas).T
homogeneidad.round(3)

In [ ]:
resumen_tests(homogeneidad, ["ANOVA", "Kruskal-Wallis", "Levene"]).round(3)

**Conclusión:** no hay diferencias significativas entre años una vez corregido por comparaciones múltiples, así que es válido juntar los tres años. Sin corregir aparecen diferencias puntuales en la media de diciembre (y marzo, sólo en ANOVA) y en la varianza de agosto.

## 5. Oferta estandarizada conjunta

Se estandariza cada día con la media y desviación de su mes y se testea la normalidad de los 1.096 valores juntos. Es el chequeo más exigente, porque usa toda la muestra, y evalúa directamente lo que el modelo simula.

In [ ]:
z = ((oferta["oferta"] - oferta["mes"].map(lambda m: normales[m].media))
     / oferta["mes"].map(lambda m: normales[m].desviacion)).to_numpy()

print(f"Shapiro-Wilk: valor-p = {stats.shapiro(z).pvalue:.2f} | "
      f"Anderson-Darling: valor-p = {normal_ad(z)[1]:.2f} | "
      f"Lilliefors: valor-p = {lilliefors(z, dist='norm')[1]:.2f}")
print(f"Asimetría = {stats.skew(z):.3f} | exceso de curtosis = {stats.kurtosis(z):.3f}")

niveles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
cuantiles = pd.DataFrame({"observado": np.quantile(z, niveles), "Normal": stats.norm.ppf(niveles)},
                         index=[f"{100 * q:.0f}%" for q in niveles])
cuantiles.index.name = "cuantil"

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
rejilla = np.linspace(-4, 4, 200)
ejes[0].hist(z, bins=30, density=True, color="0.8", edgecolor="white")
ejes[0].plot(rejilla, stats.norm.pdf(rejilla), color="C3")
ejes[0].set_title("Oferta estandarizada y Normal estándar")
teoricos, observados = stats.probplot(z, dist="norm", fit=False)
ejes[1].scatter(teoricos, observados, s=6, color="C0")
ejes[1].plot([-3.5, 3.5], [-3.5, 3.5], color="C3", lw=1)
ejes[1].set_title("Gráfico Q-Q")
ejes[1].set_xlabel("cuantil teórico")
ejes[1].set_ylabel("cuantil observado")
plt.tight_layout()
plt.show()

cuantiles.round(3)

**Conclusión:** tampoco se rechaza la normalidad con toda la muestra. Los cuantiles observados calzan con los de la Normal; las colas son levemente más cortas (exceso de curtosis de −0.28), por lo que la Normal no subestima los valores extremos.

## 6. Redondeo y truncamiento

El modelo redondea la oferta simulada a entero y la trunca en 0. Se calcula la probabilidad de que la Normal genere un valor negativo y se revisa la granularidad de los datos históricos.

In [ ]:
p_negativa = max(stats.norm.cdf(cfg.OFERTA_MINIMA - 0.5, d.media, d.desviacion) for d in normales.values())
x = oferta["oferta"].to_numpy()

print(f"Probabilidad máxima de oferta negativa en un mes: {p_negativa:.1e}")
print(f"Menor oferta histórica: {x.min()} unidades")
print(f"Días con oferta múltiplo de 3: {100 * (x % 3 == 0).mean():.0f}%")
print(f"Desviación típica dentro de un mes: {np.mean([d.desviacion for d in normales.values()]):.0f} unidades")

**Conclusión:** el truncamiento en 0 no tiene efecto (probabilidad del orden de $10^{-11}$) y el redondeo a entero es despreciable frente a una desviación de ~90 unidades. Los datos históricos son todos múltiplos de 3, mientras que el modelo genera cualquier entero; esa granularidad también es despreciable para el ajuste, pero conviene tenerla presente.

## 7. Conclusión general

- La Normal por mes no se rechaza en ningún mes con Shapiro-Wilk, Anderson-Darling ni Lilliefors (corregidos por Holm), ni tampoco con la muestra completa estandarizada.
- Los gráficos Q-Q no muestran desviaciones sistemáticas, y las colas observadas son levemente más cortas que las de la Normal.
- La Normal es la mejor distribución por AIC frente a la Gamma y la lognormal, o equivalente a la mejor.
- Los tres años son homogéneos dentro de cada mes, por lo que es válido ajustar una sola Normal por mes del año.
- El redondeo y el truncamiento en 0 no afectan.

**Recomendación:** mantener la Normal por mes del año para la oferta, sin cambios en `codigo/distribuciones.py`.